# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-araromi/flyrank-ml-internship-sulaimon/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*I chose this lane because content teams need a practical way to decide which pages deserve attention when they cannot review everything at once. The starter dataset contains observable information about search visibility, engagement, content age, freshness, and performance trends that can help identify pages worth reviewing. My project will examine how these signals can support a ranked queue of pages for possible refresh, protection, or monitoring, while keeping the final decision with a human reviewer. This direction also builds on the baseline rules and client-held-out modelling explored in the two starter notebooks.*

In [14]:
from pathlib import Path
import os
import subprocess

repo_url = (
    "https://github.com/s-araromi/"
    "flyrank-ml-internship-sulaimon.git"
)

repo_dir = Path("/content/flyrank-ml-internship-sulaimon")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_dir)],
        check=True,
    )

os.chdir(repo_dir)

starter_csv = Path("data/raw/content_refresh_anonymized.csv")

assert starter_csv.exists(), (
    f"Starter dataset was not found at: {starter_csv}"
)

print(f"Working folder: {Path.cwd()}")
print(f"Starter dataset found: {starter_csv}")

Working folder: /content/flyrank-ml-internship-sulaimon
Starter dataset found: data/raw/content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

Research question: Which visible content pages should a content strategy team prioritize for review, possible refresh, monitoring, or protection based on observable search performance, engagement, content age, and freshness signals?

Unit of analysis: One pseudonymized content page.

Decision: Determine which pages deserve attention first when the team has limited time and cannot review every page.

Who acts and what they do: A content editor, SEO analyst, or content strategist reviews the highest-ranked pages and decides whether each should be updated, monitored, protected, or left unchanged.

Expected output: A ranked review queue showing priority scores, understandable reasons for each recommendation, and suggested next actions.

Cost of a wrong recommendation: A false positive wastes editorial time on a page that does not need attention and may lead to an unnecessary update. A false negative leaves a genuinely declining or valuable page unreviewed, potentially allowing an important opportunity to be missed.

Why data or machine learning may help: Visibility, engagement, content age, freshness, and performance trends may interact in ways that are difficult to capture with a single rule. However, a transparent rule-based baseline should be established first, and a more complex model should only be used if it improves prioritization under honest validation.

Potential evaluation measure: Precision@20 or Precision@50, depending on how many pages the team can realistically review.

In [15]:
research_frame = {
    "lane": "Refresh / Content Opportunity Scoring",
    "unit_of_analysis": "one pseudonymized content page",
    "decision": "which content pages should be reviewed first",
    "decision_maker": "content editor, SEO analyst, or content strategist",
    "output": "ranked review queue with reason codes and suggested actions",
    "false_positive_cost": "unnecessary review time or an avoidable update",
    "false_negative_cost": "missing a declining or high-opportunity page",
    "evaluation_metric": "Precision@20 or Precision@50",
}

assert all(
    isinstance(value, str) and value.strip()
    for value in research_frame.values()
), "Every part of the research frame must have a clear value."

for item, description in research_frame.items():
    label = item.replace("_", " ").title()
    print(f"{label}: {description}")

Lane: Refresh / Content Opportunity Scoring
Unit Of Analysis: one pseudonymized content page
Decision: which content pages should be reviewed first
Decision Maker: content editor, SEO analyst, or content strategist
Output: ranked review queue with reason codes and suggested actions
False Positive Cost: unnecessary review time or an avoidable update
False Negative Cost: missing a declining or high-opportunity page
Evaluation Metric: Precision@20 or Precision@50


## 3. Quick look at the data (2-3 real numbers)

*The starter dataset contains 30,000 content pages across 32 pseudonymized clients. Of these pages, 16,262, representing 54.2% of the inventory, have an observed downward performance trend. An exploratory rule combining at least 500 impressions over 90 days with at least 180 days since the last update identifies 17 visible, older pages, of which 16 are declining. However, this subgroup is too small to support a stable or broadly applicable conclusion, and the 94.1% figure should be interpreted cautiously. These observations justify investigating how different combinations of visibility, freshness, and performance signals might support a practical review queue; they do not establish that refreshing a page will improve its performance.*

In [16]:
import pandas as pd

data = pd.read_csv(starter_csv)

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "days_since_last_update",
}

missing_columns = required_columns.difference(data.columns)

assert not missing_columns, (
    f"The starter dataset is missing required columns: "
    f"{sorted(missing_columns)}"
)

page_count = len(data)
client_count = data["client_id"].nunique()

declining_mask = (
    data["trend_direction"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("down")
)

declining_count = int(declining_mask.sum())
declining_percentage = 100 * declining_count / page_count

minimum_impressions = 500
minimum_days_since_update = 180

visible_and_stale_mask = (
    data["impressions_90d"].ge(minimum_impressions)
    & data["days_since_last_update"].ge(minimum_days_since_update)
)

visible_and_stale_count = int(visible_and_stale_mask.sum())

declining_visible_and_stale_count = int(
    (visible_and_stale_mask & declining_mask).sum()
)

if visible_and_stale_count > 0:
    visible_and_stale_declining_percentage = (
        100
        * declining_visible_and_stale_count
        / visible_and_stale_count
    )
else:
    visible_and_stale_declining_percentage = 0.0

print(
    f"1. Content inventory: {page_count:,} pages "
    f"across {client_count:,} pseudonymized clients."
)

print(
    f"2. Pages with an observed downward trend: "
    f"{declining_count:,} "
    f"({declining_percentage:.1f}% of all pages)."
)

print(
    f"3. Visible pages not updated for at least "
    f"{minimum_days_since_update} days: "
    f"{visible_and_stale_count:,}; "
    f"{declining_visible_and_stale_count:,} are declining "
    f"({visible_and_stale_declining_percentage:.1f}%)."
)

print(
    "\nThese figures describe observed patterns and help "
    "identify pages for human review."
)

1. Content inventory: 30,000 pages across 32 pseudonymized clients.
2. Pages with an observed downward trend: 16,262 (54.2% of all pages).
3. Visible pages not updated for at least 180 days: 17; 16 are declining (94.1%).

These figures describe observed patterns and help identify pages for human review.


## 4. Careful words: what I can and can't claim

What I can claim: The dataset contains observable differences in page visibility, content freshness, and performance trends. I can describe measured associations, compare transparent prioritization approaches, and identify pages that may warrant human review. Any ranking will be presented as decision-support evidence rather than an automatic instruction to modify content.

What I cannot claim: I cannot conclude that an outdated page caused its own decline, that refreshing a particular page will restore traffic, or that the analysis reveals how Google’s ranking algorithm works. The observed decline rate of 94.1% among 17 visible, older pages is based on a small subgroup and must not be treated as a reliable estimate for the wider inventory.

Leakage and privacy safeguards: Performance outcome fields such as trend_direction, trend_pct, and labels derived from them must not be used as model features when predicting decline. Pseudonymized client and content identifiers will be used only for grouping, tracking, or validation—not as predictive inputs. No private client names, domains, URLs, or raw search queries will be included in public outputs.

Interpretation: Any eventual model must be compared against a transparent baseline and evaluated using suitable client-held-out or time-aware validation. A recommendation remains a reason for human review, not proof of benefit or causation.

In [17]:
proposed_predictor_fields = {
    "impressions_90d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "word_count",
}

excluded_from_predictors = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
}

unsafe_overlap = (
    proposed_predictor_fields & excluded_from_predictors
)

assert not unsafe_overlap, (
    "Potential leakage or identifier misuse detected: "
    f"{sorted(unsafe_overlap)}"
)

minimum_cautious_group_size = 30

print("Leakage check: passed.")
print(
    "Outcome fields and pseudonymized identifiers "
    "are excluded from proposed predictors."
)

print(
    f"Exploratory visible-and-stale subgroup: "
    f"{visible_and_stale_count} pages."
)

if visible_and_stale_count < minimum_cautious_group_size:
    print(
        "Caution: this subgroup is small; "
        "its observed decline percentage is unstable "
        "and should not be generalized."
    )

print(
    "Interpretation: observed associations and "
    "decision support only; no causal claims."
)

Leakage check: passed.
Outcome fields and pseudonymized identifiers are excluded from proposed predictors.
Exploratory visible-and-stale subgroup: 17 pages.
Caution: this subgroup is small; its observed decline percentage is unstable and should not be generalized.
Interpretation: observed associations and decision support only; no causal claims.


## Self-check

Before you submit, confirm each line honestly:

## Self-check

- [x] Every section is completed with clear explanations and supporting code.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, private URLs, domains, or raw search queries appear.
- [x] Claims are observational, cautious, and intended for decision support.
- [x] The executed notebook has been committed to `work/notebooks/`.